# Figure 2 — Spatial and temporal distribution of traffic exposure, weather conditions, and crashes

- **FIG_ID**: `Figure 2`（编号真源：`data/figure_numbering.csv`，§4.0；output stem `Figure_2`）
- **图位**: v6 正文行 189（§4.1）；对应 `[FIGURE 2 PLACEHOLDER]`
- **版面**: 固定四 panel（§1.6.8）：(a) study area map；(b) temporal distribution；
  (c) exposure distributions；(d) exposure–response summary
- **数据依赖**（A 层冻结，SHA-256 见 `data/figure_data_manifest.json`）:
  - `data/fig02_monthly.csv`（(b) 月度事故计数 + 月度平均流量，同窗口 pooled，含 z 列）
  - `data/fig02_distributions.csv`（(c) 四种天气暴露逐断面日分布，27,400 行）
  - `data/fig02_elasticity_curve.csv`（(d) 弹性曲线格点 + delta 法点态 95% 带）
  - `data/fig02_elasticity_anchors.csv`（(d) P10–P90 锚点 + 2.44/2.68 线性参照）
  - `data/fig02_elasticity_meta.csv`（(d) 模型系数/协方差/窗口）
  - `figure/zhuyong_expressway.png`（(a) 仅作走廊几何来源：亮绿廊道像素提取中心线后
    自绘简化示意图；卫星底图/中文地名/逐桩编号不进图，2026-10-02 总体判断 P0 裁决）
- **上游变更提示**: 若 B 层主面板 / analysis_data_v2 或 A 层 T131/T132 更新，先运行
  `python scripts/extract_fig02_data.py` 重新冻结（含 15 条逐值断言），再执行本册
  （Cell 3 的 `verify_figure_data()` 会拦截哈希不匹配）
- **输出**: `out/draft/Figure_2.{png,pdf,eps}`（`FIG_DRAFT=1`）或
  `out/submission/Figure_2.{png,pdf,eps}`（`FIG_DRAFT=0`）+ `out/captions/Figure_2_caption.txt`


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time
from pathlib import Path


def _resolve_scripts_dir() -> Path:
    """定位 figure/scripts，使本册在**任何内核 cwd** 下都可运行。

    兼容场景：cwd = figure/notebooks（nbconvert 默认）、figure/、仓库根（VS Code / 任意
    启动位置的默认行为）、以及更上层目录；最后回退到本仓库的绝对路径，保证一定能找到
    fig_common.py（旧版写死 `Path.cwd().parent / "scripts"`，只在 notebooks/ 下可用）。
    """
    here = Path.cwd()
    cands = [here / "scripts", here.parent / "scripts"]
    cands += [p / "figure" / "scripts" for p in (here, *here.parents)]
    cands.append(Path(r"D:\NJU_folder\final_update\15_paper\figure\scripts"))
    for c in cands:
        if (c / "fig_common.py").exists():
            return c.resolve()
    raise RuntimeError("未找到 figure/scripts/fig_common.py：请从本仓库、figure/ 或 "
                       "figure/notebooks/ 任一目录启动内核，或检查仓库路径")


SCRIPTS_DIR = _resolve_scripts_dir()
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))
print("scripts dir:", SCRIPTS_DIR)

from fig_common import *          # 样式、色板、导出、核验（唯一来源，勿在本册硬编码样式）
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f">>> Figure 2 notebook | design system {DESIGN_VERSION} | notebook API v{NOTEBOOK_API} "
      f"| 四 panel + NJU purple")
print(">>> 若本行未出现、或版本号低于上面的值，说明内核里跑的是**旧版 notebook**："
      "请在编辑器执行 Revert File（VS Code）/ 关闭标签后重新打开本文件")

# ---- 本图配置 -------------------------------------------------------------
FIG_ID = "Figure 2"
IS_DRAFT = os.environ.get("FIG_DRAFT", "1") == "1"   # 1 -> out/draft；0 -> out/submission
DATA_FILES = ["fig02_monthly.csv", "fig02_distributions.csv", "fig02_elasticity_curve.csv",
              "fig02_elasticity_anchors.csv", "fig02_elasticity_meta.csv"]
t0 = time.time()
meta = figure_meta(FIG_ID)
print(f"FIG_ID={FIG_ID}  stem={meta['filename_stem']}  IS_DRAFT={IS_DRAFT}")
print("data dir   :", FIG_DATA, "| 数据文件齐备:", all((FIG_DATA / f).exists() for f in DATA_FILES))
print("输出目录:", FIG_OUT_DRAFT if IS_DRAFT else FIG_OUT_SUB)


In [ ]:
# ---- 数据加载与核验（不匹配即中断）---------------------------------------
report = verify_figure_data(FIG_ID, data_files=DATA_FILES)
monthly = pd.read_csv(FIG_DATA / "fig02_monthly.csv")
dist = pd.read_csv(FIG_DATA / "fig02_distributions.csv")
curve = pd.read_csv(FIG_DATA / "fig02_elasticity_curve.csv")
anchors = pd.read_csv(FIG_DATA / "fig02_elasticity_anchors.csv")
emeta = pd.read_csv(FIG_DATA / "fig02_elasticity_meta.csv").set_index("key")["value"]
assert (FIGURE_DIR / "zhuyong_expressway.png").exists(), "缺少 panel (a) 走廊轮廓图"

# 结构性自检（防止冻结文件被替换成"形似而数不同"的内容）
assert len(monthly) == 12 and list(monthly["month"]) == list(range(1, 13))
assert len(dist) == 27400, "核心面板分布取数应为 27,400 行（2023–2025）"
assert set(dist.columns) == {"precip_sum_mm", "visibility_min_m", "wind_gust_max_ms", "freezing_hours"}
assert len(curve) == 200
_needed = {"flow_value", "log_flow", "local_elasticity_E", "se_delta", "ci_low", "ci_high"}
assert _needed.issubset(curve.columns), (
    f"fig02_elasticity_curve.csv 缺少列 {sorted(_needed - set(curve.columns))}；"
    "请先运行 python figure\\scripts\\extract_fig02_data.py 重新冻结")
anch = anchors[anchors.row_type == "anchor"]
lref = anchors[anchors.row_type == "linear_ref"]
assert list(anch["key"]) == ["P10", "P25", "P50", "P75", "P90"]
assert set(lref["key"]) == {"linear_length_offset", "linear_volume_offset"}
# 登记值核对（计划书 §2-Fig2：锚点 1.5107/1.9235/2.3249/2.725/3.105）
assert np.allclose(anch["local_elasticity_E"], [1.5107, 1.9235, 2.3249, 2.7250, 3.1050], atol=1e-3)
_len = lref.set_index("key").loc["linear_length_offset"]
assert abs(_len["estimate"] - 2.44) < 5e-3 and abs(_len["ci_low"] - 2.10) < 5e-3
assert abs(lref.set_index("key").loc["linear_volume_offset", "estimate"] - 2.68) < 1e-9
assert int(float(emeta["n_estimation"])) == 25599
print("核验通过:", report["checked"])
print()
print(pd.DataFrame({
    "值": ["核心面板行数", "(b) 年事故计数合计（全窗口）", "(d) 估计样本 n",
           "(d) 曲线窗口（P1–P99, veh/day）", "(d) 锚点 E（P10→P90）",
           "线性 length-offset 参照", "线性 volume-offset 参照"],
    "取值": [f"{len(dist):,}", f"{int(monthly.crash_count.sum()):,}",
             f"{int(float(emeta['n_estimation'])):,}",
             f"{float(emeta['grid_flow_p1']):,.0f} – {float(emeta['grid_flow_p99']):,.0f}",
             " / ".join(f"{v:.3f}" for v in anch['local_elasticity_E']),
             f"{_len['estimate']:.3f} [{_len['ci_low']:.3f}, {_len['ci_high']:.3f}]",
             "2.680 (interval not frozen)"]}).to_string(index=False))


In [ ]:
# ---- 绘图逻辑：四 panel 固定布局 ------------------------------------------
# 面板装配代码放在同图模块 fig02_distribution_core.build_figure() 内，
# 使命令行试绘（--draft/--submission）与本册出图共用唯一实现，避免两处拷贝漂移。
from fig02_distribution_core import build_figure

fig = build_figure()
for ax in fig.get_axes():
    print(f"{ax.get_label():>6s}  xlim={tuple(round(float(v), 2) for v in ax.get_xlim())}  "
          f"ylim={tuple(round(float(v), 2) for v in ax.get_ylim())}")
fig


In [ ]:
# ---- 导出双版本 + caption + qc -------------------------------------------
written = export_figure(fig, FIG_ID, is_draft=IS_DRAFT)
print("\n".join(str(p) for p in written))

from fig02_distribution_core import CAPTION

cap = write_caption(FIG_ID, CAPTION,
                    title="Figure 2. Spatial and temporal distribution of traffic exposure, "
                          "weather conditions, and crashes.")
print(cap)

qc = fig_qc_dump(FIG_ID, {
    "status": "ok", "is_draft": IS_DRAFT, "seconds": round(time.time() - t0, 1),
    "data_checked": report["checked"], "data_files": DATA_FILES,
    "outputs": [{"path": str(p), "bytes": p.stat().st_size, "sha256": sha256_file(p)} for p in written],
    "caption": str(cap),
})
print(qc)

# 投稿版凭据：只在非草稿运行时写入，避免被后续草稿重跑覆盖（发布门禁以它为准）
if not IS_DRAFT:
    rec = fig_submission_record(FIG_ID, written, cap,
                                {"data_checked": report["checked"], "data_files": DATA_FILES})
    print("submission record:", rec)
